In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

transactions = spark.table(
    "workspace.pyspark_deep_dive.transactions"
)

customers = spark.table(
    "workspace.pyspark_deep_dive.customers"
)

products = spark.table(
    "workspace.pyspark_deep_dive.products"
)

enriched_transactions = spark.table(
    "workspace.pyspark_deep_dive.enriched_transactions"
)

print("Transactions:", transactions.count())
print("Customers:", customers.count())
print("Products:", products.count())
print("Enriched:", enriched_transactions.count())

In [0]:
from pyspark.sql import functions as F

df = spark.table(
    "workspace.pyspark_deep_dive.enriched_transactions"
)

filtered_df = (
    df
    .filter(F.col("total_amount") > 5000)
    .select(
        "transaction_id",
        "customer_id",
        "total_amount"
    )
)

print("Transformation created")

In [0]:
display(filtered_df.limit(10))

In [0]:
filtered_df.explain("formatted")

In [0]:
customer_spend = (
    df
    .groupBy("customer_id")
    .agg(
        F.sum("total_amount").alias("total_spend")
    )
)

customer_spend.explain("formatted")

In [0]:
sorted_df = (
    df
    .orderBy(F.col("total_amount").desc())
)

sorted_df.explain("formatted")

In [0]:
display(sorted_df.limit(20))

In [0]:
sorted_df.explain("formatted")

In [0]:
from pyspark.sql.functions import broadcast

broadcast_join = (
    transactions.alias("t")
    .join(
        broadcast(products).alias("p"),
        F.col("t.product_id") == F.col("p.product_id"),
        "inner"
    )
)

broadcast_join.explain("formatted")

In [0]:
skewed_df = (
    spark.range(1, 1_000_001)
    .withColumn(
        "customer_id",
        F.when(
            F.col("id") <= 500_000,
            F.lit(999)
        ).otherwise(
            (F.rand(seed=500) * 10_000).cast("int")
        )
    )
)

display(
    skewed_df
    .groupBy("customer_id")
    .count()
    .orderBy(F.col("count").desc())
    .limit(10)
)

In [0]:
salted_df = skewed_df.withColumn('salt', F.floor(F.rand(seed=501) * 10))\
.withColumn('salted_customer_id', F.concat(
    F.col('customer_id').cast('string'), F.lit('_'), F.col('salt').cast('string')
))

In [0]:
display(salted_df.groupBy("salted_customer_id").count().orderBy(F.col("count").desc()).limit(15))